# Verify and compare the theoretical 3-core transitions

This notebook evaluates the deterministic calculations in notebooks 05 and 08.
It checks their derivatives and square-root coefficients, displays the critical
values, and exports the table and figure used in Chapter 3. It generates no
random graphs and reads no simulation measurements.

The shared theory notebooks remain safe to load with `%run`: they define
functions without writing files or displaying plots. Run this notebook from top
to bottom, or use `make theory-figures` from the project root. Source notebook
outputs are deliberately left empty; the command executes a fresh kernel without
saving an executed notebook copy.

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

%run "{NOTEBOOK_DIR}/05_configuration_theory.ipynb"
%run "{NOTEBOOK_DIR}/08_triangle_cavity.ipynb"

PROJECT_ROOT = NOTEBOOK_DIR.parent

In [ ]:
import csv

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

## Output settings

These paths contain deterministic theory products, not graph measurements.
`report_dir` holds the selected paper assets; the other diagnostic exports stay
under `results/`. Re-running this notebook replaces only these named theory
products. The source notebooks and simulation CSVs are never modified.

In [ ]:
config = {
    "report_dir": "report",
    "diagnostic_dir": "results/figures/theory",
    "delta_min": 1e-7,  # Smaller offsets eventually amplify root roundoff.
    "delta_max": 0.08,  # Show where higher-order terms become visible.
    "curve_points": 160,  # Deterministic plotting resolution, not simulation size.
}

## Critical values and density coefficients

For each model the positive branch has the expansion

$$
\rho_M(\lambda_c+\delta)=\rho_c+a_M\sqrt\delta+O(\delta),
\qquad \delta\downarrow0.
$$

The coefficient $a_M$ describes the **core density**, not the message coordinate.
The two agree only after multiplying the message displacement by the derivative
of the density map. The triangle values remain predictions of its cavity
equations. The CM values refer to the retained multigraph.

We also evaluate the CM fluctuation scale, including both degree sampling and
edge pairing. It is expressed in units of the membership parameter $\lambda$;
it is not the square-root density coefficient.

In [ ]:
cm = cm_fold_expansion()
triangle = triangle_fold()
critical_rows = [
    {
        "model": "CM multigraph",
        "lambda_c": cm["lambda"],
        "rho_c": cm["rho"],
        "a": cm["a"],
    },
    {
        "model": "Triangle cavity",
        "lambda_c": triangle["lambda"],
        "rho_c": triangle["rho"],
        "a": triangle["a"],
    },
]
cm_noise = cm_fluctuation_constants(critical=cm)
cm_noise

In [ ]:
table_markdown = (
    r"| Model | $\lambda_c$ | $\rho_c$ | $a$ |" + "\n|---|---:|---:|---:|\n"
)
for row in critical_rows:
    table_markdown += (
        f"| {row['model']} | {row['lambda_c']:.8f} | "
        f"{row['rho_c']:.8f} | {row['a']:.8f} |\n"
    )
display(Markdown(table_markdown))

## Independent numerical checks

Each model's verification functions compare analytic partial derivatives with
centred differences at several step sizes. They then solve the original branch
at decreasing positive offsets and report

$$
\frac{\rho_M(\lambda_c+\delta)-\rho_c}{\sqrt\delta}.
$$

This ratio must approach the independently differentiated $a_M$. It should not
be forced to equal $a_M$ at a fixed nonzero offset: the expansion has an
$O(\delta)$ remainder, producing an $O(\sqrt\delta)$ error in the ratio. Very tiny
offsets eventually magnify roundoff in a nearly double root. The chosen range
stays above that regime.

The exported diagnostics show the step sizes and offsets, rather than hiding
this conditioning issue behind one arbitrarily small tolerance.

In [ ]:
cm_derivatives = cm_fold_derivative_checks(critical=cm)
cm_branches = cm_fold_branch_checks(critical=cm)
triangle_checks = triangle_fold_checks(critical=triangle)
derivative_rows = []
for row in cm_derivatives:
    derivative_rows.append(
        {
            "model": "CM multigraph",
            "step": row["step"],
            "G_lambda_error": row["G_lambda_difference"] - cm["G_lambda"],
            "G_yy_error": row["G_xx_difference"] - cm["G_xx"],
            "rho_y_error": row["rho_x_difference"] - cm["rho_x"],
            "a_relative_error": row["a_relative_error"],
        }
    )
for row in triangle_checks["derivatives"]:
    numerical_a = row["rho_z"] * np.sqrt(-2 * row["H_lambda"] / row["H_zz"])
    derivative_rows.append(
        {
            "model": "Triangle cavity",
            "step": row["step"],
            "G_lambda_error": row["H_lambda_error"],
            "G_yy_error": row["H_zz_error"],
            "rho_y_error": row["rho_z_error"],
            "a_relative_error": abs(numerical_a / triangle["a"] - 1),
        }
    )
branch_rows = []
for row in cm_branches:
    delta = row["offset"]
    branch_rows.append(
        {
            "model": "CM multigraph",
            "delta": delta,
            "rho": row["rho"],
            "density_ratio": row["density_ratio"],
            "a": cm["a"],
            "ratio_error": row["density_ratio_error"],
            "remainder_over_delta": (row["rho"] - row["rho_approximation"]) / delta,
            "residual": row["residual"],
        }
    )
for row in triangle_checks["branch"]:
    branch_rows.append(
        {
            "model": "Triangle cavity",
            "delta": row["delta"],
            "rho": row["rho"],
            "density_ratio": row["density_ratio"],
            "a": triangle["a"],
            "ratio_error": row["density_ratio"] - triangle["a"],
            "remainder_over_delta": row["remainder_over_delta"],
            "residual": row["fixed_point_residual"],
        }
    )
check_table = "| Model | Smallest offset | Density ratio | Derived coefficient |\n|---|---:|---:|---:|\n"
for name in ["CM multigraph", "Triangle cavity"]:
    differences = [row for row in derivative_rows if row["model"] == name]
    roots = [row for row in branch_rows if row["model"] == name]
    assert min(row["a_relative_error"] for row in differences) < 1e-6
    assert abs(roots[-1]["ratio_error"]) < abs(roots[0]["ratio_error"]) / 100
    assert max(row["residual"] for row in roots) < 1e-10
    last = roots[-1]
    check_table += f"| {name} | {last['delta']:.0e} | {last['density_ratio']:.8f} | {last['a']:.8f} |\n"
display(Markdown(check_table))

## Check the CM matching-noise constant independently

The compact pairing-variance formula in notebook 05 is obtained by summing the
covariances in Janson--Łuczak (2008), Eq. (5.40). The following check evaluates
that original finite double sum and its integrals directly. It does not use the
simplified covariance formula inside the integral.

Let $q_j(u)=\Pr(\operatorname{Bin}(K,u)\geq j)$ and use
$q_j'(u)=\mathbb E[K\Pr(\operatorname{Bin}(K-1,u)=j-1)]$. With weights $w_3=3$
and $w_r=1$ for $r>3$, the reference covariance is

$$
\sigma_{WH}(p)=\sum_{r=3}^\infty w_r\,2p^{r+2}
 \sum_{j=r}^\infty {j-1\choose r-1}
 \int_p^1 (u-p)^{j-r}u^{-j-2}q_j'(u)\,du.
$$

Here $p=x_c$. We truncate the Poisson membership sum at 21 (degree 42),
well beyond the contribution visible at double precision, and compare three
Gauss--Legendre quadrature resolutions. The result must agree with
$4\lambda_c p^2(1-p^2)$. We also compare the variance function's membership
cutoffs 40 and 60. These checks involve probability sums and integration,
not a Monte Carlo estimate.

In [ ]:
from math import comb, exp, factorial

p, lam = cm["x"], cm["lambda"]
degree_probabilities = {2 * d: exp(-lam) * lam**d / factorial(d) for d in range(22)}
covariance_rows = []
for points in [48, 96, 192]:
    nodes, weights = np.polynomial.legendre.leggauss(points)
    u = p + (1 - p) * (nodes + 1) / 2
    weights = weights * (1 - p) / 2
    max_degree = max(degree_probabilities)
    tail_derivatives = {
        j: sum(
            k * probability * comb(k - 1, j - 1) * u ** (j - 1) * (1 - u) ** (k - j)
            for k, probability in degree_probabilities.items()
            if k >= j
        )
        for j in range(3, max_degree + 1)
    }
    covariance = 0.0
    for r in range(3, max_degree + 1):
        weight_r = 3 if r == 3 else 1
        for j in range(r, max_degree + 1):
            integral = np.dot(
                weights, (u - p) ** (j - r) * u ** (-j - 2) * tail_derivatives[j]
            )
            covariance += weight_r * 2 * p ** (r + 2) * comb(j - 1, r - 1) * integral
    simplified = 4 * lam * p**2 * (1 - p**2)
    covariance_rows.append(
        {
            "quadrature_points": points,
            "direct_covariance": covariance,
            "closed_covariance": simplified,
            "absolute_error": abs(covariance - simplified),
        }
    )
assert max(row["absolute_error"] for row in covariance_rows) < 1e-12
noise_60 = cm_fluctuation_constants(critical=cm, max_memberships=60)
for key in ["variance_matching", "variance_degree", "tau"]:
    assert abs(cm_noise[key] - noise_60[key]) < 1e-13
covariance_rows

## Theory curves and the local square-root approximation

The upper panels show the maximal positive branch and its separate zero branch,
with a vertical dotted line indicating the predicted jump. The value at the
critical point is the positive branch's right-hand limit; it is not a
prediction that a finite graph at exactly $\lambda_c$ has that deterministic
size. The lower panels expose the local approximation error by dividing out the
square root. Their horizontal dashed line is the derivative-based coefficient,
not a fit to the plotted curve.

In [ ]:
offsets = np.geomspace(config["delta_min"], config["delta_max"], config["curve_points"])
curves = []
figure, axes = plt.subplots(2, 2, figsize=(7.1, 4.5), layout="constrained")
for column, (label, fold, prediction, color) in enumerate(
    [
        ("CM multigraph", cm, cm_prediction, "#217075"),
        ("Triangle cavity", triangle, triangle_prediction, "#b67031"),
    ]
):
    values = np.array(
        [prediction(fold["lambda"] + delta, critical=fold)["rho"] for delta in offsets]
    )
    parameter = fold["lambda"] + offsets
    approximation = fold["rho"] + fold["a"] * np.sqrt(offsets)
    ratios = (values - fold["rho"]) / np.sqrt(offsets)
    for delta, rho, approx, ratio in zip(offsets, values, approximation, ratios):
        curves.append(
            {
                "model": label,
                "delta": float(delta),
                "lambda": float(fold["lambda"] + delta),
                "rho": float(rho),
                "square_root": float(approx),
                "ratio": float(ratio),
            }
        )
    top, bottom = axes[:, column]
    top.plot(parameter, values, color=color, label="Equation solution", linewidth=1.5)
    top.plot(
        parameter,
        approximation,
        color="black",
        linestyle="--",
        label="Square-root approximation",
        linewidth=1.0,
    )
    top.plot(
        [fold["lambda"] - 0.025, fold["lambda"]], [0, 0], color=color, linewidth=1.5
    )
    top.plot(
        [fold["lambda"], fold["lambda"]],
        [0, fold["rho"]],
        color=color,
        linestyle=":",
        linewidth=1.0,
    )
    top.plot(
        fold["lambda"],
        0,
        "o",
        markerfacecolor="white",
        markeredgecolor=color,
        markersize=4,
    )
    top.plot(fold["lambda"], fold["rho"], "o", color=color, markersize=4)
    top.set(
        title=label,
        xlabel=r"Membership parameter $\lambda$",
        ylabel=r"Core fraction $\rho$",
    )
    top.legend(fontsize=7, loc="upper left", frameon=False)
    bottom.semilogx(offsets, ratios, color=color, linewidth=1.5)
    bottom.axhline(
        fold["a"],
        color="black",
        linestyle="--",
        linewidth=1.0,
        label=rf"Derivative value $a={fold['a']:.4f}$",
    )
    bottom.set(
        xlabel=r"Distance above threshold $\delta$",
        ylabel=r"$(\rho(\lambda_c+\delta)-\rho_c)/\sqrt{\delta}$",
    )
    bottom.legend(fontsize=7, frameon=False)
    for axis in (top, bottom):
        axis.tick_params(labelsize=8)
        axis.xaxis.label.set_size(9)
        axis.yaxis.label.set_size(9)
        axis.title.set_size(10)
        axis.spines[["top", "right"]].set_visible(False)
        axis.grid(alpha=0.16, linewidth=0.5)
figure

## Export the paper assets and numerical provenance

The CSV contains more digits for reproducible checks. The paper table rounds to
six decimals, which describes numerical solutions of the stated equations, not
physical finite-graph accuracy. The triangle row is labelled as a cavity
prediction. The figure is a vector PDF, so its lines and labels remain sharp
in the two-column report.

In [ ]:
report_dir = PROJECT_ROOT / config["report_dir"]
diagnostic_dir = PROJECT_ROOT / config["diagnostic_dir"]
(report_dir / "tables").mkdir(parents=True, exist_ok=True)
(report_dir / "figures").mkdir(parents=True, exist_ok=True)
diagnostic_dir.mkdir(parents=True, exist_ok=True)


def save_theory_csv(path, rows):
    """Write named deterministic results with a header; no experiment data."""
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


save_theory_csv(diagnostic_dir / "critical_values.csv", critical_rows)
save_theory_csv(diagnostic_dir / "density_curves.csv", curves)
save_theory_csv(diagnostic_dir / "derivative_checks.csv", derivative_rows)
save_theory_csv(diagnostic_dir / "branch_checks.csv", branch_rows)
save_theory_csv(diagnostic_dir / "cm_fluctuation_constants.csv", [cm_noise])
save_theory_csv(diagnostic_dir / "cm_covariance_checks.csv", covariance_rows)
figure.savefig(
    report_dir / "figures/theory_fold_comparison.pdf",
    metadata={"CreationDate": None, "ModDate": None},
)
figure.savefig(diagnostic_dir / "theory_fold_comparison.png", dpi=180)
latex_rows = [
    "% Generated by notebooks/theory_comparison.ipynb; regenerate with make theory-figures.",
    r"\begin{tabular}{lrrr}",
    r"\toprule",
    r"Model & $\lambda_c$ & $\rho_c$ & $a$ \\",
    r"\midrule",
]
for row in critical_rows:
    latex_rows.append(
        f"{row['model']} & {row['lambda_c']:.6f} & {row['rho_c']:.6f} & {row['a']:.6f} "
        + r"\\"
    )
latex_rows.extend([r"\bottomrule", r"\end{tabular}"])
(report_dir / "tables/theory_critical_values.tex").write_text(
    "\n".join(latex_rows) + "\n", encoding="utf-8"
)
print("Updated theory table, figure, and diagnostic CSVs.")

## What these calculations establish

The density coefficients and curves are consequences of the deterministic
fixed-point equations. The CM fluctuation calculation additionally uses the
configuration-model pruning theorem and random-degree conditioning argument;
it includes both sources of Gaussian noise. Its constants do not automatically
transfer to the erased CM control or to triangle communities.

The proposed triangle window and conditional distribution remain hypotheses.
Chapter 3 states them using a random effective threshold. Later experiments
will estimate its centre and width from survival indicators alone, then test the
predicted conditional core sizes without refitting a density location,
amplitude, or spread. No fitting or random graph simulation occurs here.